In [3]:
from langgraph.graph import StateGraph,START,END
from langchain_huggingface import ChatHuggingFace,HuggingFaceEndpoint

In [4]:
from typing import TypedDict

In [5]:
from kaggle_secrets import UserSecretsClient
secret_label = "your-secret-label"
secret_value = UserSecretsClient().get_secret('HUGGINGFACEHUB_API_TOKEN')

In [6]:
model=HuggingFaceEndpoint(repo_id='mistralai/Mistral-7B-Instruct-v0.2',
    temperature=0.1,
    max_new_tokens=2048,
                          task='conversational',
    huggingfacehub_api_token=secret_value)

In [7]:
llm=ChatHuggingFace(llm=model)

In [8]:
class BlogState(TypedDict):
    title: str
    outline: str
    content: str

In [9]:
def create_outline(state: BlogState) -> BlogState:
    title = state['title']
    prompt = f'Generate a detailed outline for a blog on the topic - {title}'
    outline = llm.invoke(prompt).content
    state['outline'] = outline
    return state

In [10]:
def create_blog(state: BlogState) -> BlogState:
    title = state['title']
    outline = state['outline']
    prompt = f'Write a detailed blog on the title - {title} using the follwing outline \n {outline}'
    content = llm.invoke(prompt).content
    state['content'] = content
    return state

In [11]:
graph = StateGraph(BlogState)
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)

In [12]:
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', END)

In [13]:
workflow = graph.compile()

In [14]:
a= workflow.invoke({'title':'The dominance of China in ai'})

In [15]:
print(a['outline'])

 Title: "The Dominance of China in AI: A New Tech Superpower"

I. Introduction

* Brief overview of the current state of Artificial Intelligence (AI) and its significance in the modern world
* Mention of China's rapid rise in the AI sector and its potential implications

II. China's Investment in AI

* Discussion of China's significant investment in AI research and development
* Analysis of government initiatives, such as "Made in China 2025" and the "Next Generation Artificial Intelligence Development Plan"
* Overview of private sector investments and collaborations

III. China's AI Research and Development

* Description of key research institutions and universities leading AI research in China
* Analysis of China's strengths in specific areas of AI, such as computer vision, natural language processing, and robotics
* Comparison of China's research output and achievements with those of other leading AI nations

IV. China's AI Applications and Use Cases

* Discussion of how China is a

In [16]:
print(a['content'])

 Title: The Dominance of China in AI: A New Tech Superpower

I. Introduction

Artificial Intelligence (AI) has become a significant force in the modern world, transforming industries, creating new business opportunities, and driving technological innovation. One country that has been making waves in the AI sector is China. With its rapid rise in AI research and development, China is poised to become a major player in this field and potentially redefine the global tech landscape.

II. China's Investment in AI

China's investment in AI is a testament to its commitment to becoming a global leader in this technology. According to a report by Tsinghua University and the Elsevier, China became the world's largest investor in AI research and development in 2019, with a total investment of $32 billion. This investment is driven by both government initiatives and private sector collaborations.

The Chinese government has launched several initiatives to boost AI research and development. One suc

In [17]:
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field
import operator

In [18]:
class EvaluationSchema(BaseModel):
    feedback: str = Field(description='Detailed feedbackfor the essay')
    Score: int = Field(description='Score out of 100', ge=0, le=100)

In [19]:
from langchain_core.output_parsers import PydanticOutputParser

In [20]:
prs=PydanticOutputParser(pydantic_object=EvaluationSchema)

In [21]:
class UPSCState(TypedDict):
    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individual_scores: Annotated[list[int], operator.add]
    avg_score: float

In [22]:
from langchain_core.prompts import PromptTemplate

In [23]:
def evaluate_language(state: UPSCState):
    tem=PromptTemplate(template='''Evaluate the language quality of the following essay and provide a feedback and assign a score out of 100 \n {essay} \nReturn ONLY valid JSON.
Do NOT include markdown.
Do NOT include explanations outside the JSON.
Return exactly this structure:

"feedback": string,
"Score": integer
{form}''',
                   input_variables=['essay'],
                   partial_variables={'form':prs.get_format_instructions()})
    strt=tem|llm|prs
    output = strt.invoke({'essay':state["essay"]})

    return {'language_feedback': output.feedback, 'individual_scores': [output.Score]}

In [24]:
def evaluate_analysis(state: UPSCState):
    tem=PromptTemplate(template='''Evaluate the depth of analysis of the following essay and provide a feedback and assign a score out of 100 \n {essay} \nReturn ONLY valid JSON.
Do NOT include markdown.
Do NOT include explanations outside the JSON.
Return exactly this structure:

"feedback": string,
"Score": integer
{form}''',
                   input_variables=['essay'],
                   partial_variables={'form':prs.get_format_instructions()})
    strt=tem|llm|prs
    output = strt.invoke({'essay':state["essay"]})

    return {'analysis_feedback': output.feedback, 'individual_scores': [output.Score]}

In [25]:
def evaluate_thought(state: UPSCState):
    tem=PromptTemplate(template='''Evaluate the clarity of thought of the following essay and provide a feedback and assign a score out of 100 \n {essay} \nReturn ONLY valid JSON.
Do NOT include markdown.
Do NOT include explanations outside the JSON.
Return exactly this structure:

"feedback": string,
"Score": integer
{form}''',
                   input_variables=['essay'],
                   partial_variables={'form':prs.get_format_instructions()})
    strt=tem|llm|prs
    output = strt.invoke({'essay':state["essay"]})

    return {'clarity_feedback': output.feedback, 'individual_scores': [output.Score]}

In [26]:
def final_evaluation(state: UPSCState):
    prompt = f'Based on the following feedbacks create a summarized feedback \n language feedback - {state["language_feedback"]} \n depth of analysis feedback - {state["analysis_feedback"]} \n clarity of thought feedback - {state["clarity_feedback"]}'
    overall_feedback = llm.invoke(prompt).content
    avg_score = sum(state['individual_scores'])/len(state['individual_scores'])
    return {'overall_feedback': overall_feedback, 'avg_score': avg_score}

In [27]:
graph1 = StateGraph(UPSCState)

In [28]:
graph1.add_node('evaluate_language', evaluate_language)
graph1.add_node('evaluate_analysis', evaluate_analysis)
graph1.add_node('evaluate_thought', evaluate_thought)
graph1.add_node('final_evaluation', final_evaluation)

In [29]:
graph1.add_edge(START, 'evaluate_language')
graph1.add_edge(START, 'evaluate_analysis')
graph1.add_edge(START, 'evaluate_thought')
graph1.add_edge('evaluate_language', 'final_evaluation')
graph1.add_edge('evaluate_analysis', 'final_evaluation')
graph1.add_edge('evaluate_thought', 'final_evaluation')
graph1.add_edge('final_evaluation', END)

In [30]:
b=graph1.compile()

In [31]:
b.invoke({'essay':a['content']})

{'essay': ' Title: The Dominance of China in AI: A New Tech Superpower\n\nI. Introduction\n\nArtificial Intelligence (AI) has become a significant force in the modern world, transforming industries, creating new business opportunities, and driving technological innovation. One country that has been making waves in the AI sector is China. With its rapid rise in AI research and development, China is poised to become a major player in this field and potentially redefine the global tech landscape.\n\nII. China\'s Investment in AI\n\nChina\'s investment in AI is a testament to its commitment to becoming a global leader in this technology. According to a report by Tsinghua University and the Elsevier, China became the world\'s largest investor in AI research and development in 2019, with a total investment of $32 billion. This investment is driven by both government initiatives and private sector collaborations.\n\nThe Chinese government has launched several initiatives to boost AI research 